## Shared datasets

Run the setup cells, then the section for the dataset you want to obtain or update. Every dataset is saved in its own descriptively named folder. This notebook does not execute projects or import figure generators.


## Failure reporting
Each operation reports what failed, the exception reason, and what to assess. Unexpected failures retain their traceback and stop execution. A missing project dataset is handled separately with a debug message and skipped dependent calculations.

In [ ]:
from contextlib import contextmanager
import subprocess

@contextmanager
def report_failure(stage, assessment):
    try:
        yield
    except Exception as error:
        print(f"FAILED: {stage}", flush=True)
        print(f"Reason: {type(error).__name__}: {error}", flush=True)
        if isinstance(error, subprocess.CalledProcessError):
            details = error.stderr or error.stdout or error.output
            if details:
                if isinstance(details, bytes):
                    details = details.decode("utf-8", errors="replace")
                print(f"Process output (last 4000 characters):\n{str(details)[-4000:]}", flush=True)
        print(f"Assess: {assessment}", flush=True)
        print("This operation failed. Correct the cause and rerun it before dependent cells.", flush=True)
        raise


## Dataset libraries

Install the pinned libraries used for source retrieval, normalization and inventory displays.


In [ ]:
with report_failure('Dataset libraries', 'Check network access, package availability, Python compatibility, installation permissions and available disk space.'):
    # Purpose: install the pinned standard libraries used by this one pipeline.
    import importlib.metadata
    import subprocess
    import sys
    import shutil
    REQUIRED_PACKAGES = ('nbformat==5.11.1','ipykernel==7.4.0','numpy==2.3.5','scipy==1.17.0','matplotlib==3.10.8','huggingface-hub==0.36.2','scikit-learn==1.8.0','safetensors==0.8.0')
    def install_notebook_dependencies(packages=REQUIRED_PACKAGES):
        missing=[]
        for package in packages:
            name,expected=package.split('==',1)
            try: installed=importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError: installed=None
            if installed!=expected: missing.append(package)
        if missing:
            subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],check=True)
    install_notebook_dependencies()


## Dataset storage

Set DATASETS_ROOT to the persistent location you use for shared data. Files are updated in the same dataset folder. On Colab, mount persistent storage or save the downloaded files before the runtime ends.


In [ ]:
with report_failure('Dataset storage', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    from pathlib import Path
    import os
    import json
    ROOT = Path(os.environ.get("ADVANCED_DATA_SCIENCE_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "Datasets").is_dir() and (candidate / "project_notebooks").is_dir():
            ROOT = candidate
            break
    PROJECT_DIRECTORY = ROOT
    DATASETS_ROOT = Path(os.environ.get("DATASETS_ROOT", str(ROOT / "Datasets"))).expanduser().resolve()
    DATASETS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Dataset destination: {DATASETS_ROOT}")


## Corpus source and reproducibility

**Dataset:** [Arxiv_2020_OCR on Hugging Face](https://huggingface.co/datasets/SlowGuess/Arxiv_2020_OCR).

**Pinned revision:** `a68009864dbc3b1decb1a37a6c4f769887169a44`.

This existing release provides arXiv documents identified by their 2020 identifiers. The inspected document directory contains its original PDF, extracted full text in `full.md`, and page-layout and content records. The corpus comprises every document directory in this pinned release; there is no document-count cap, random subsampling, class balancing by deletion, or filtering by extraction quality.

The publisher's card says “OCR Data” but does not document the complete extraction or correction procedure. The notebook uses the released full-text bytes directly and records their checksums. It performs no PDF extraction, optical character recognition, or text reconstruction.

**Original scientific-document source:** [arXiv](https://arxiv.org).

**Additional source record preserved for peer verification:** [Cornell University arXiv dataset on Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv), with [arXiv bulk access documentation](https://info.arxiv.org/help/bulk_data.html). Kaggle's downloadable metadata is distinct from the full PDFs provided through its linked public collection.

The earlier news-corpus results are historical computations and do not describe this corpus.

## Corpus identity

Define the dataset repository, immutable revision and expected source count that govern inventory and download verification.


In [ ]:
with report_failure('Corpus identity', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    # Source identity for extracted arXiv research-paper texts.
    DATASET_REPOSITORY = "SlowGuess/Arxiv_2020_OCR"
    DATASET_SOURCE_URL = "https://huggingface.co/datasets/SlowGuess/Arxiv_2020_OCR"
    DATASET_REVISION = "a68009864dbc3b1decb1a37a6c4f769887169a44"
    ORIGINAL_SOURCE_URL = "https://arxiv.org"
    KAGGLE_SOURCE_URL = "https://www.kaggle.com/datasets/Cornell-University/arxiv"
    MINIMUM_PUBLICATION_YEAR = 2020
    # Preserve every document in this release, including missing or problematic extractions.
    # The publisher does not specify the upstream correction policy. Preserve released bytes.


## Libraries and output locations

Import the numerical and display libraries and establish corpus, result and figure directories beneath the project directory.


In [ ]:
with report_failure('Libraries and output locations', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    from pathlib import Path
    import hashlib
    import json
    import csv
    import numpy as np
    from scipy import sparse
    import matplotlib.pyplot as plt
    from IPython.display import display, Markdown

    EXPERIMENT_DIRECTORY = DATASETS_ROOT / "arxiv_research_papers_2020"
    EXPERIMENT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    RESULTS_DIRECTORY = EXPERIMENT_DIRECTORY / "diagnostics"
    RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    FIGURE_DIRECTORY = RESULTS_DIRECTORY / "figures"
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({"figure.dpi": 140, "font.size": 10})

    import io
    from IPython.display import Image

    def display_figure(figure, name):
        image_buffer = io.BytesIO()
        figure.savefig(image_buffer, format="png", bbox_inches="tight")
        (FIGURE_DIRECTORY / name).write_bytes(image_buffer.getvalue())
        display(Image(data=image_buffer.getvalue()))
        plt.close(figure)


## Complete source inventory

Read or obtain the pinned dataset inventory. Retain every source and record the assets needed to verify the downloaded text.


In [ ]:
with report_failure('Complete source inventory', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    from huggingface_hub import HfApi
    from pathlib import PurePosixPath
    from collections import defaultdict
    import re

    SOURCE_DIRECTORY = EXPERIMENT_DIRECTORY
    SOURCE_DIRECTORY.mkdir(exist_ok=True)
    manifest_path = SOURCE_DIRECTORY / "source_manifest.json"
    if manifest_path.exists():
        source_manifest = json.loads(manifest_path.read_text())
        assert source_manifest["repository"] == DATASET_REPOSITORY
        assert source_manifest["revision"] == DATASET_REVISION
    else:
        repository = HfApi().dataset_info(DATASET_REPOSITORY, revision=DATASET_REVISION, files_metadata=True)
        assert repository.sha == DATASET_REVISION
        documents = defaultdict(dict)
        for asset in repository.siblings:
            path = PurePosixPath(asset.rfilename)
            identifier = path.parent.name
            if not re.fullmatch(r"20(?:0[1-9]|1[0-2])\.\d{4,5}", identifier):
                assert path.name in {"README.md", ".gitattributes"}, f"Unaccounted repository asset: {path}"
                continue
            entry = documents[identifier]
            entry["document_id"] = identifier
            entry["source_directory"] = str(path.parent)
            entry.setdefault("assets", []).append({
                "path": asset.rfilename,
                "bytes": asset.size,
                "git_blob_sha1": asset.blob_id,
                "sha256": getattr(asset.lfs, "sha256", None) if asset.lfs else None,
            })
        source_manifest = {
            "repository": DATASET_REPOSITORY,
            "source_url": DATASET_SOURCE_URL,
            "revision": DATASET_REVISION,
            "membership_rule": "Every document directory in the complete pinned repository",
            "repository_file_count": len(repository.siblings),
            "documents": [documents[key] for key in sorted(documents)],
        }
        manifest_path.write_text(json.dumps(source_manifest, indent=2))

    source_documents = source_manifest["documents"]
    assert source_documents
    assert len({record["document_id"] for record in source_documents}) == len(source_documents)
    source_document_count = len(source_documents)
    text_file_count = sum(any(PurePosixPath(a["path"]).name == "full.md" for a in record["assets"]) for record in source_documents)
    pdf_file_count = sum(any(a["path"].endswith(".pdf") for a in record["assets"]) for record in source_documents)
    text_bytes = sum(a["bytes"] or 0 for record in source_documents for a in record["assets"] if PurePosixPath(a["path"]).name == "full.md")
    display(Markdown(f"**{source_document_count:,} source documents**; {text_file_count:,} extracted full-text files; {pdf_file_count:,} documents with original PDFs. Full-text file size: {text_bytes:,} bytes."))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    values = [source_document_count, text_file_count, pdf_file_count]
    bars = ax.bar(["Source documents", "Extracted full text", "Original PDF available"], values, color=["#456990", "#078078", "#456990"])
    ax.bar_label(bars, labels=[f"{value:,}" for value in values], padding=4)
    ax.set_ylim(0, max(values)*1.15)
    ax.set_ylabel("Documents")
    ax.set_title("Complete pinned arXiv corpus inventory")
    fig.tight_layout()
    display_figure(fig, "source_inventory.png")


## Verified source texts

Use the inventory to obtain the extracted text, checking its length and checksum. Reuse verified files and retain all source documents.


In [ ]:
with report_failure('Verified source texts', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    import urllib.request
    import urllib.error
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from datetime import datetime, timezone
    from email.utils import parsedate_to_datetime
    from threading import Lock
    import math

    # One shared request schedule prevents concurrent workers from multiplying the rate.
    SOURCE_REQUEST_LOCK = Lock()
    NEXT_SOURCE_REQUEST_AT = [0.0]
    SOURCE_REQUEST_INTERVAL = 0.2

    RAW_TEXT_DIRECTORY = SOURCE_DIRECTORY / "original_text"
    RAW_TEXT_DIRECTORY.mkdir(exist_ok=True)

    def save_json(path, value):
        temporary = path.with_name(path.name + ".tmp")
        temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
        temporary.replace(path)

    def verify_source_bytes(data, asset):
        assert len(data) == asset["bytes"], f"Incomplete source file: {asset['path']}"
        if asset["sha256"]:
            assert hashlib.sha256(data).hexdigest() == asset["sha256"]
        else:
            header = f"blob {len(data)}\0".encode()
            assert hashlib.sha1(header + data).hexdigest() == asset["git_blob_sha1"]
        data.decode("utf-8", errors="strict")

    def wait_for_source_request():
        while True:
            with SOURCE_REQUEST_LOCK:
                now = time.monotonic()
                remaining = NEXT_SOURCE_REQUEST_AT[0] - now
                if remaining <= 0:
                    NEXT_SOURCE_REQUEST_AT[0] = now + SOURCE_REQUEST_INTERVAL
                    return
            time.sleep(min(remaining, 60.0))


    def defer_source_requests(headers, attempt, status=None):
        # A 429 pauses every worker. Honor both Retry-After formats and use a
        # bounded exponential delay when the source omits that header.
        delay = min((60.0 if status == 429 else 2.0) * 2 ** attempt, 480.0)
        retry_after = (headers or {}).get("Retry-After", "")
        if retry_after:
            try:
                header_delay = float(retry_after)
            except (TypeError, ValueError):
                try:
                    retry_date = parsedate_to_datetime(retry_after)
                    if retry_date.tzinfo is None:
                        retry_date = retry_date.replace(tzinfo=timezone.utc)
                    header_delay = (retry_date - datetime.now(timezone.utc)).total_seconds()
                except (TypeError, ValueError, OverflowError):
                    header_delay = 0.0
            if math.isfinite(header_delay):
                delay = max(delay, header_delay)
        with SOURCE_REQUEST_LOCK:
            NEXT_SOURCE_REQUEST_AT[0] = max(NEXT_SOURCE_REQUEST_AT[0], time.monotonic() + delay)
        return delay


    def fetch_source_bytes(url):
        for attempt in range(8):
            wait_for_source_request()
            try:
                request = urllib.request.Request(url, headers={"User-Agent": "ResearchCorpus/1.0"})
                with urllib.request.urlopen(request, timeout=60) as response:
                    return response.read()
            except urllib.error.HTTPError as error:
                if error.code not in {408, 429, 500, 502, 503, 504} or attempt == 7:
                    raise
                defer_source_requests(error.headers, attempt, error.code)
            except (OSError, urllib.error.URLError):
                if attempt == 7:
                    raise
                defer_source_requests({}, attempt)
        raise RuntimeError("Source download exhausted its retry policy.")


    def obtain_existing_text(record):
        assets = [asset for asset in record["assets"] if asset["path"].endswith("/full.md")]
        assert len(assets) == 1, f"Expected one existing full text: {record['document_id']}"
        asset = assets[0]
        destination = RAW_TEXT_DIRECTORY / (record["document_id"] + ".md")
        if destination.exists():
            data = destination.read_bytes()
            verify_source_bytes(data, asset)
        else:
            url = f"https://huggingface.co/datasets/{DATASET_REPOSITORY}/resolve/{DATASET_REVISION}/{asset['path']}"
            data = fetch_source_bytes(url)
            verify_source_bytes(data, asset)
            temporary = destination.with_suffix(".part")
            temporary.write_bytes(data)
            temporary.replace(destination)
        return {"document_id": record["document_id"], "source_path": asset["path"],
                "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(),
                "git_blob_sha1": asset["git_blob_sha1"]}

    receipts, download_errors = [], []
    started = time.monotonic()
    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = {executor.submit(obtain_existing_text, record): record["document_id"] for record in source_documents}
        for future in as_completed(futures):
            identifier = futures[future]
            try:
                receipts.append(future.result())
            except Exception as error:
                download_errors.append({"document_id": identifier, "error": str(error)})
            completed = len(receipts) + len(download_errors)
            if completed % 100 == 0 or completed == len(source_documents):
                progress = {"source_documents": len(source_documents), "verified_text_files": len(receipts),
                            "errors": download_errors, "elapsed_seconds": time.monotonic() - started}
                save_json(SOURCE_DIRECTORY / "download_progress.json", progress)
                print(f"Verified {len(receipts):,}/{len(source_documents):,} complete text files; errors: {len(download_errors)}", flush=True)

    save_json(SOURCE_DIRECTORY / "source_text_register.json", {
        "repository": DATASET_REPOSITORY, "revision": DATASET_REVISION,
        "documents": sorted(receipts, key=lambda row: row["document_id"]), "errors": download_errors,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "processing": "Existing full.md bytes copied without alteration. No PDF download or optical character recognition."})
    assert not download_errors, "Retry unresolved downloads; no source document may be silently dropped."
    assert {row["document_id"] for row in receipts} == {row["document_id"] for row in source_documents}
    assert sum(row["bytes"] for row in receipts) == text_bytes
    print(f"All {len(receipts):,} documents retained byte-for-byte: {text_bytes:,} bytes.")


## Publication dates and target labels

Associate the retained arXiv identifiers with publication metadata and primary categories. These categories provide the classification targets.


In [ ]:
with report_failure('Publication dates and target labels', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    import urllib.parse
    import urllib.request
    import xml.etree.ElementTree as ET
    import time

    METADATA_DIRECTORY = SOURCE_DIRECTORY / "arxiv_metadata"
    METADATA_DIRECTORY.mkdir(exist_ok=True)
    metadata_path = SOURCE_DIRECTORY / "document_metadata.json"
    metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
    identifiers = [record["document_id"] for record in source_documents]
    missing = [identifier for identifier in identifiers if identifier not in metadata]
    atom_namespace = {"atom": "http://www.w3.org/2005/Atom", "arxiv": "http://arxiv.org/schemas/atom"}
    for start in range(0, len(missing), 100):
        batch = missing[start:start + 100]
        request_key = hashlib.sha256("\n".join(batch).encode()).hexdigest()
        response_path = METADATA_DIRECTORY / (request_key + ".xml")
        if response_path.exists():
            raw_xml = response_path.read_bytes()
        else:
            query = urllib.parse.urlencode({"id_list": ",".join(batch), "max_results": len(batch)})
            url = "https://export.arxiv.org/api/query?" + query
            for attempt in range(4):
                try:
                    with urllib.request.urlopen(url, timeout=90) as response:
                        raw_xml = response.read()
                    ET.fromstring(raw_xml)
                    response_path.write_bytes(raw_xml)
                    break
                except (OSError, ET.ParseError):
                    if attempt == 3:
                        raise
                    time.sleep(3 * (attempt + 1))
            time.sleep(3.1)
        returned = set()
        for entry in ET.fromstring(raw_xml).findall("atom:entry", atom_namespace):
            versioned_id = entry.findtext("atom:id", default="", namespaces=atom_namespace).rsplit("/", 1)[-1]
            identifier = re.sub(r"v\d+$", "", versioned_id)
            assert identifier in batch, f"Unexpected metadata record: {identifier}"
            published = entry.findtext("atom:published", default="", namespaces=atom_namespace)
            primary = entry.find("arxiv:primary_category", atom_namespace)
            assert primary is not None and published.startswith("2020-"), (identifier, published)
            metadata[identifier] = {
                "document_id": identifier,
                "published": published,
                "primary_category": primary.attrib["term"],
                "title": " ".join(entry.findtext("atom:title", default="", namespaces=atom_namespace).split()),
                "doi": entry.findtext("arxiv:doi", default="", namespaces=atom_namespace),
                "metadata_versioned_id": versioned_id,
                "metadata_response_file": response_path.name,
                "metadata_response_sha256": hashlib.sha256(raw_xml).hexdigest(),
            }
            returned.add(identifier)
        assert returned == set(batch), "Resolve missing metadata without removing documents."
        temporary_metadata = metadata_path.with_suffix(".tmp")
        temporary_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
        temporary_metadata.replace(metadata_path)
        print(f"Verified source labels and dates for {len(metadata):,}/{len(identifiers):,} documents.", flush=True)
    assert set(metadata) == set(identifiers)
    print("Publication dates and primary arXiv categories verified for the complete corpus.")


## Text normalization and coverage

Normalize every verified text while recording character changes and source correspondence. Report document counts and length distributions before chunking.


In [ ]:
with report_failure('Text normalization and coverage', 'Check the reported source URL or file path, network access, available disk space and destination permissions. Preserve verified files; rerun this dataset cell after correcting the cause.'):
    import unicodedata
    from collections import Counter

    NORMALIZED_TEXT_DIRECTORY = SOURCE_DIRECTORY / "normalized_text"
    NORMALIZED_TEXT_DIRECTORY.mkdir(exist_ok=True)
    NORMALIZATION_DIRECTORY = SOURCE_DIRECTORY / "normalization_records"
    NORMALIZATION_DIRECTORY.mkdir(exist_ok=True)

    def normalize_text(text):
        pieces, edits = [], []
        operation_counts = Counter()
        source_position = normalized_position = 0
        while source_position < len(text):
            char = text[source_position]
            source_end = source_position + 1
            operation = None
            if char == "\r":
                source_end += int(source_end < len(text) and text[source_end] == "\n")
                replacement, operation = "\n", "line_ending_to_lf"
            elif char == "\t":
                replacement, operation = " ", "tab_to_space"
            elif char == "\n":
                replacement = char
            elif unicodedata.category(char) == "Zs":
                replacement = " "
                if char != replacement:
                    operation = "horizontal_space_to_space"
            elif not char.isprintable():
                replacement, operation = "", "nonprinting_removed"
            else:
                replacement = char.lower()
                if replacement != char:
                    operation_counts["lowercased_characters"] += 1
                if len(replacement) != 1:
                    operation = "lowercase_character_expansion"
            if operation:
                edits.append({"source_start": source_position, "source_end": source_end,
                              "normalized_start": normalized_position,
                              "normalized_end": normalized_position + len(replacement),
                              "operation": operation})
                operation_counts[operation] += 1
            pieces.append(replacement)
            normalized_position += len(replacement)
            source_position = source_end
        normalized = "".join(pieces)
        assert normalized == normalized.lower()
        assert all(char == "\n" or char.isprintable() for char in normalized)
        return normalized, edits, dict(operation_counts)

    normalization_records = []
    for number, record in enumerate(source_documents, 1):
        identifier = record["document_id"]
        raw_path = RAW_TEXT_DIRECTORY / (identifier + ".md")
        asset = next(asset for asset in record["assets"] if asset["path"].endswith("/full.md"))
        raw = raw_path.read_bytes()
        verify_source_bytes(raw, asset)
        source_text = raw.decode("utf-8", errors="strict")
        normalized_path = NORMALIZED_TEXT_DIRECTORY / (identifier + ".txt")
        record_path = NORMALIZATION_DIRECTORY / (identifier + ".json")
        source_digest = hashlib.sha256(raw).hexdigest()
        if record_path.exists() and normalized_path.exists():
            saved = json.loads(record_path.read_text())
            normalized = normalized_path.read_text()
            assert saved["source_sha256"] == source_digest
            assert hashlib.sha256(normalized.encode()).hexdigest() == saved["normalized_sha256"]
        else:
            normalized, edits, counts = normalize_text(source_text)
            normalized_path.write_text(normalized, encoding="utf-8")
            saved = {"document_id": identifier, "source_sha256": source_digest,
                     "normalized_sha256": hashlib.sha256(normalized.encode()).hexdigest(),
                     "source_characters": len(source_text), "normalized_characters": len(normalized),
                     "words": len(normalized.split()), "empty": not bool(normalized.strip()),
                     "operations": counts, "offset_edits": edits}
            save_json(record_path, saved)
        normalization_records.append({key: value for key, value in saved.items() if key != "offset_edits"})
        if number % 250 == 0:
            print(f"Normalized {number:,}/{len(source_documents):,} complete documents.", flush=True)
    assert {row["document_id"] for row in normalization_records} == {row["document_id"] for row in source_documents}
    save_json(SOURCE_DIRECTORY / "normalization_manifest.json", {
        "source_revision": DATASET_REVISION, "document_count": len(normalization_records),
        "rule": "Lowercase; retain visible characters, ordinary spaces and line feeds; normalize line endings and horizontal spaces; remove other nonprinting characters.",
        "document_filtering": False, "text_truncation": False, "semantic_repair": False,
        "documents": normalization_records})
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    counts = [len(source_documents), len(normalization_records)]
    bars = axes[0].bar(["Source documents", "Normalized documents"], counts, color="#078078")
    axes[0].bar_label(bars, padding=3)
    axes[0].set_ylim(0, max(counts)*1.15)
    axes[0].set_ylabel("Documents")
    axes[0].set_title("Complete document retention")
    lengths = np.array([row["words"] for row in normalization_records])
    axes[1].hist(lengths, bins=60, color="#456990")
    axes[1].set_xlabel("Words per complete normalized document")
    axes[1].set_ylabel("Documents")
    axes[1].set_title("Document length distribution")
    fig.tight_layout()
    display_figure(fig, "normalization_coverage_and_lengths.png")
    print(f"Retained all {len(normalization_records):,} documents, including {sum(row['empty'] for row in normalization_records):,} empty texts.")
